In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()
from loguru import logger

In [2]:
fs = sorted(Path("../data/json2").glob("*.json"))
print(f"{len(fs)} threads")

10257 threads


### Extract titles from markdown links

In [3]:
# We will get a list of title from markdown links
import re
from collections import defaultdict

titles_md = defaultdict(list)


title_blocklist = [
    "patreon",
    "here",
    "link",
    "this",
    "delete",
    "this one",
    "vote",
    "image",
    "comic explanation",
    "this link",
    "this thread",
    "redact",
    "(link)",
    "this post",
    "CLICK THIS LINK",
    "here's",
    "wiki",
    "report",
    "source",
    "the wiki",
    "reddit - dive into anything",
    "top posts",
    "a patreon campaign",
    "table of contents",
    "fanfiction.net",
    "this video",
    "here.",
    'tubmlr',
    "threads",
    "code",
    "link index",
    "some discussions",
    "for",
    "details",
    "pushshift",
    "Reddit - Dive into anything",
    "table of contents",
    "rationalreads",
    "calibre",
    "goodreads",
]

title_should_not_have = [
    "vote",
    "comments",
    "link",
    "here",
    "image",
    "chapter",
    "reddit",
    "wiki",
    "pateron",
    "source",
    'chapter',
    'amazon',
    'kindle',
    'epub',
]

def get_md_link_titles(markdown_text):
    # other rules
    # 1. should have space
    # not subreddit?
    # should have capital
    titles_md2 = {}

    # Regular expression to match markdown links
    markdown_link_pattern = re.compile(r"\[([^\]]+)\]\((http[s]?://[^\)]+)\)")

    # Find all markdown links
    for match in markdown_link_pattern.findall(markdown_text):
        title, url = match
        titles_md2[url] = []
        # tidy title of * / _ and remove leading/trailing whitespace
        title = title.strip().strip("*").strip("_")

        if "r/" in title:
            continue

        # I mean ideally it has spaces but Worm doesn't hpmor doesn't
        # if " " not in title:
        #     continue

        # should have caps
        if title.lower() == title:
            continue

        # should have at least 4 characters e,g, Worm, HPMOR
        if len(title) < 4:
            continue

        if title.startswith("http"):
            continue
        if title.startswith("^"):
            continue

        if title.lower() in title_blocklist:
            continue

        if any([x in title.lower() for x in title_should_not_have]):
            continue

        titles_md2[url].append(title)

    return titles_md2


def remember_md_link_titles(markdown_text):

    md_title_dict = get_md_link_titles(markdown_text)
    for k, v in md_title_dict.items():
        if 'http' not in v:
            titles_md[k].extend(v)

for f in tqdm(fs):
    s = json.loads(f.open().read())

    for comment in s["comments"]:
        text = comment["body"]
        remember_md_link_titles(text)

len(titles_md)

  0%|          | 0/10257 [00:00<?, ?it/s]

29313

In [4]:
titles_md

defaultdict(list,
            {'https://readcomiconline.li/Comic/Squarriors-2014': [],
             'https://readcomiconline.li/Comic/Alt-Life': ['Alt-life'],
             'https://readcomiconline.li/Comic/Pride-of-Baghdad': ['Pride of Baghdad'],
             'https://readcomiconline.li/Comic/%C3%86ther-Empire': ['Aether and empire'],
             'https://readcomiconline.li/Comic/Guinea-Pigs': ['Guinea pigs'],
             'https://readcomiconline.li/Comic/920London/TPB?id=172150': ['920London'],
             'https://www.lackadaisycats.com/': ['Lackadaisy Cats',
              'Lackadaisy Cats'],
             'https://mangadex.org/title/ad42a9f3-8e8e-405e-88be-2ca2bbe4847b/child': ['Child'],
             'https://mangadex.org/title/4f665ec1-a93a-46e4-936d-54bfde53063b/child-4koma': ['Child 4koma'],
             'https://readcomiconline.li/Comic/Crashing': ['Crashing'],
             'https://readcomiconline.li/Comic/Eight-Billion-Genies': ['Eight Billion Genies'],
             'https:/

In [5]:
# def get_title_md(url):
#     t = titles_md.get(url, [])
#     if t:
#         s = pd.Series(t)
#         return s.value_counts().index[0]  # return most common title
#     return url
# get_title_md('https://parahumans.wordpress.com/')

In [6]:
# QC: get top markdown titles
import itertools

l = sorted(itertools.chain(*titles_md.values()))
pd.Series(l).value_counts().head(50)

Time Braid                                     93
Mother of Learning                             66
Worm                                           66
Worth the Candle                               64
Twig                                           58
Statistics                                     52
Problems/Bugs?                                 52
Stop Replying                                  52
Pact                                           36
With This Ring                                 36
The Waves Arisen                               34
Marked for Death                               34
The Metropolitan Man                           33
Luminosity                                     32
What is this?                                  32
Pale                                           31
A Hero's War                                   30
Unsong                                         30
Friendship is Optimal                          30
Dungeon Keeper Ami                             29


### Extract links

In [7]:
def extract_links_re(s: str):
    return re.findall(r"\[.*?\]\((.*?)\)", s)


from collections import defaultdict

# TODO consolidate into a metadata object
link_metadata = {
    "karma": defaultdict(int),  # track score
    "gossip": defaultdict(list),  # track number and content of comments...
    "gossip_threads": defaultdict(list),  # threads
    "dates": defaultdict(list),  # comment dates
}


def stem_url(u):
    parts = u.strip().rstrip("/").split("/")
    return "/".join(parts[:-1])


# banned_link_suffixes = ['jpeg', 'png', 'jpg']

data = []
for f in tqdm(fs):
    post = json.loads(f.open().read())
    thread_url = "https://reddit.com" + post["permalink"]
    thread_created = post["created_utc"]

    for comment in post["comments"]:
        text = comment["body"]
        links = extract_links_re(text)
        for link in links:
            data.append(
                dict(
                    url=link,
                    score=comment["score"],
                    # created_utc=post["created_utc"],
                    # comment_url="https://reddit.com" + comment["permalink"],
                    # comment_body=comment["body"],
                    comment=comment,
                    thread_url=thread_url,
                )
            )

print(f"{len(data)} links found")

  0%|          | 0/10257 [00:00<?, ?it/s]

46353 links found


In [8]:
df_links1 = pd.DataFrame(data)
# check for dups
# print(df_links['url'].value_counts().head(10))

# df_links1["created_utc"] = pd.to_datetime(df_links1["created_utc"], unit="s")


def join_uniq_list(x: list) -> list:
    return list(set(x))

# def joun_uniq_comment(comments: list) -> list:
#     d = {x['permalink']: x for x in comments}
#     return list(d.values())


df_links2 = df_links1.groupby("url").agg(
    score=("score", "sum"),
    n_links=("score", "count"),
    comments=("comment", list),
    thread_urls=("thread_url", join_uniq_list),
).sort_values("score", ascending=False)

# order comments by date
df_links2["comments"] = df_links2["comments"].apply(
    lambda x: sorted(x, key=lambda y: y["created_utc"])
)

# now add come things like first and last comment time
df_links2['n_comments'] = df_links2['comments'].apply(len)
df_links2['first_link_utc'] = pd.to_datetime(df_links2['comments'].apply(lambda x: min([y['created_utc'] for y in x])), unit='s')
df_links2['last_link_utc'] = pd.to_datetime(df_links2['comments'].apply(lambda x: max([y['created_utc'] for y in x])), unit='s')
df_links2

,score,n_links,comments,thread_urls,n_comments,first_link_utc,last_link_utc
url,,,,,,,
https://www.patreon.com/alexanderwales,1408,22,"[{'id': 'cqgom57', 'created_utc': 1429380827.0...",[https://reddit.com/r/rational/comments/mtlce4...,22,2015-04-18 18:13:47,2021-04-29 20:00:04
https://archiveofourown.org/works/11478249/chapters/25740126,794,63,"[{'id': 'dluq8vo', 'created_utc': 1503171093.0...",[https://reddit.com/r/rational/comments/dwrzbr...,63,2017-08-19 19:31:33,2024-04-17 19:03:01
https://www.fanfiction.net/s/10360716/1/The-Metropolitan-Man,663,55,"[{'id': 'chvfe2o', 'created_utc': 1401507549.0...",[https://reddit.com/r/rational/comments/134n9q...,55,2014-05-31 03:39:09,2023-06-26 19:21:24
https://discord.gg/sM99CF3,646,60,"[{'id': 'd88ddo9', 'created_utc': 1475248463.0...",[https://reddit.com/r/rational/comments/7k08ef...,60,2016-09-30 15:14:23,2018-08-10 17:12:02
https://docs.google.com/document/d/1EUSMDHdRdbvQJii5uoSezbjtvJpxdF6Da8zqvuW42bg/edit?usp=sharing,627,58,"[{'id': 'd80h9fe', 'score': 4, 'body': 'So a w...",[https://reddit.com/r/rational/comments/7k08ef...,58,2016-09-24 19:53:40,2018-08-10 17:12:02
...,...,...,...,...,...,...,...
https://www.reddit.com/r/rational/comments/ix25dk/d_monday_request_and_recommendation_thread/g64soul/,-9,1,"[{'id': 'g65db9r', 'created_utc': 1600722605.0...",[https://reddit.com/r/rational/comments/ix25dk...,1,2020-09-21 21:10:05,2020-09-21 21:10:05
https://reddit.com/message/compose/?to=FatFingerHelperBot&subject=delete&message=delete%20e68m7xq,-10,1,"[{'id': 'e68m7xq', 'score': -10, 'body': 'It s...",[https://reddit.com/r/rational/comments/9h1454...,1,2018-09-19 04:51:09,2018-09-19 04:51:09
https://reddit.com/message/compose/?to=FatFingerHelperBot&subject=delete&message=delete%20dxgv0pj,-11,1,"[{'id': 'dxgv0pj', 'created_utc': 1523905663.0...",[https://reddit.com/r/rational/comments/8cppyk...,1,2018-04-16 19:07:43,2018-04-16 19:07:43


In [9]:
print(f"{len(df_links2)} unique links found")

33292 unique links found


#### Filter links

In [10]:
banned_link_suffixes = ["jpeg", "png", "jpg"]

link_blocklist = [
    #   'reddit',
    "redact",
    "pastebin",
    "wikipedia",
    "docs.google",
    "discord",
    "tvtropes.org",
    "ask_wikibot",
    "autowiki",
    "banned",
    # pateron.com ?
    "reddit.com/user/",
    # 'smile.amazon.com',
    "xkcd",
    'feedly.com',
    'autohotkey.com',
    "ebay",
    "youtubot",
    "reddit.com/r/rational",
    # 'sneakpeekbot', 'RemindMeBot',
    # 'WikiSummarizerBot', 'bot/', 'Bot/',
    "knowyourmeme.com",
    "UserSim",
    "vote.php",
    "youtube",
    "github",
    "imgur",
    "wikisummarizer",
    "mozilla.org",
    "reddit.com/message",
    "autotldr",
    "/top/",
    "redd.it",
    "reddit.com/u",
    "fanficfare",
    "bot.com",
    "greasyfork",
    'edit?', # google docs
]


link_allowlist = ["hfy"]

df_links3 = df_links2.copy()

print(f"{len(df_links3)} links before blocklist")
df_links3 = df_links3[
    ~df_links3.index.str.contains("|".join(link_blocklist), regex=True)
    | df_links3.index.str.contains("|".join(link_allowlist), regex=True)
]
print(f"{len(df_links3)} links after blocklist")
df_links3 = df_links3[df_links3.index.str.startswith("http")]
for suffix in banned_link_suffixes:
    df_links3 = df_links3[~df_links3.index.str.endswith(suffix)]
print(f"{len(df_links3)} links after rm suffixes")

# # must have more than one mention?
# df_links3 = df_links3[df_links3 > 1]
# print(f"{len(df_links3)} after count")

# if it has reddit, github, wiki and bot in the title, it's probably a bot
df_links3 = df_links3[
    ~(
        df_links3.index.str.contains("reddit", case=False)
        & df_links3.index.str.contains("bot", case=False)
    )
]
df_links3 = df_links3[
    ~(
        df_links3.index.str.contains("wiki", case=False)
        & df_links3.index.str.contains("bot", case=False)
    )
]
df_links3 = df_links3[
    ~(
        df_links3.index.str.contains("github", case=False)
        & df_links3.index.str.contains("bot", case=False)
    )
]
print(f"{len(df_links3)} links after rm bot")
df_links3


33292 links before blocklist
21635 links after blocklist
17914 links after rm suffixes
17888 links after rm bot


,score,n_links,comments,thread_urls,n_comments,first_link_utc,last_link_utc
url,,,,,,,
https://www.patreon.com/alexanderwales,1408,22,"[{'id': 'cqgom57', 'created_utc': 1429380827.0...",[https://reddit.com/r/rational/comments/mtlce4...,22,2015-04-18 18:13:47,2021-04-29 20:00:04
https://archiveofourown.org/works/11478249/chapters/25740126,794,63,"[{'id': 'dluq8vo', 'created_utc': 1503171093.0...",[https://reddit.com/r/rational/comments/dwrzbr...,63,2017-08-19 19:31:33,2024-04-17 19:03:01
https://www.fanfiction.net/s/10360716/1/The-Metropolitan-Man,663,55,"[{'id': 'chvfe2o', 'created_utc': 1401507549.0...",[https://reddit.com/r/rational/comments/134n9q...,55,2014-05-31 03:39:09,2023-06-26 19:21:24
https://www.fictionpress.com/s/2961893/1/Mother-of-Learning,549,60,"[{'id': 'cj87tis', 'created_utc': 1406362872.0...",[https://reddit.com/r/rational/comments/j9fac4...,60,2014-07-26 08:21:12,2021-03-15 18:12:29
https://twigserial.wordpress.com/,439,48,"[{'id': 'cy39n9f', 'score': 11, 'body': 'Outsi...",[https://reddit.com/r/rational/comments/f1rj3l...,48,2015-12-18 10:22:59,2024-09-11 20:51:31
...,...,...,...,...,...,...,...
https://www.merriam-webster.com/dictionary/not%20nearly,-6,1,"[{'id': 'gfaoohc', 'created_utc': 1607618149.0...",[https://reddit.com/r/rational/comments/kaat79...,1,2020-12-10 16:35:49,2020-12-10 16:35:49
https://wiki.lesswrong.com/wiki/Orthogonality_thesis,-8,2,"[{'id': 'cxn65fy', 'score': 0, 'body': 'I don'...",[https://reddit.com/r/rational/comments/3vc0si...,2,2015-12-04 18:17:07,2016-07-11 16:27:30
https://news.harvard.edu/gazette/story/2018/11/when-starting-school-younger-children-are-more-likely-to-be-diagnosed-with-adhd-study-says/,-9,1,"[{'id': 'easqwb3', 'created_utc': 1543598269.0...",[https://reddit.com/r/rational/comments/a1t6um...,1,2018-11-30 17:17:49,2018-11-30 17:17:49


In [11]:
# QC look at removed links
df_links2[~df_links2.index.isin(df_links3.index)].sort_values("score", ascending=False)

,score,n_links,comments,thread_urls,n_comments,first_link_utc,last_link_utc
url,,,,,,,
https://discord.gg/sM99CF3,646,60,"[{'id': 'd88ddo9', 'created_utc': 1475248463.0...",[https://reddit.com/r/rational/comments/7k08ef...,60,2016-09-30 15:14:23,2018-08-10 17:12:02
https://docs.google.com/document/d/1EUSMDHdRdbvQJii5uoSezbjtvJpxdF6Da8zqvuW42bg/edit?usp=sharing,627,58,"[{'id': 'd80h9fe', 'score': 4, 'body': 'So a w...",[https://reddit.com/r/rational/comments/7k08ef...,58,2016-09-24 19:53:40,2018-08-10 17:12:02
https://www.youtube.com/watch?v=kbyTOAlhRHk,440,42,"[{'id': 'de5sdvj', 'created_utc': 1487951990.0...",[https://reddit.com/r/rational/comments/7k08ef...,42,2017-02-24 15:59:50,2020-05-25 21:14:35
https://docs.google.com/document/d/11QAh61C8gsL-5KbdIy5zx3IN6bv_E9UkHjwMLVQ7LHg/edit?usp=sharing,430,42,"[{'id': 'ddv88im', 'created_utc': 1487348445.0...",[https://reddit.com/r/rational/comments/7k08ef...,42,2017-02-17 16:20:45,2018-08-10 17:12:02
https://redact.dev/home,422,79,"[{'id': 'cyzo8d7', 'score': 2, 'body': 'politi...",[https://reddit.com/r/rational/comments/gkw30p...,79,2016-01-15 21:18:35,2020-11-04 03:12:20
...,...,...,...,...,...,...,...
https://en.wikipedia.org/wiki/Gaming_disorder,-9,1,"[{'id': 'easqwb3', 'created_utc': 1543598269.0...",[https://reddit.com/r/rational/comments/a1t6um...,1,2018-11-30 17:17:49,2018-11-30 17:17:49
https://www.reddit.com/r/rational/comments/ix25dk/d_monday_request_and_recommendation_thread/g64soul/,-9,1,"[{'id': 'g65db9r', 'created_utc': 1600722605.0...",[https://reddit.com/r/rational/comments/ix25dk...,1,2020-09-21 21:10:05,2020-09-21 21:10:05
https://reddit.com/message/compose/?to=FatFingerHelperBot&subject=delete&message=delete%20e68m7xq,-10,1,"[{'id': 'e68m7xq', 'score': -10, 'body': 'It s...",[https://reddit.com/r/rational/comments/9h1454...,1,2018-09-19 04:51:09,2018-09-19 04:51:09


In [12]:
# df_links.plot.hist(bins=25, logy=True)

# Fetch missing titles

This is hard as it's an adverserial web scraping problem, I'll use a mix of methods (from the markdown, requests, url)

In [95]:
from anycache import anycache

f_cache = Path("../outputs/.anycache")
f_cache_web = Path("../outputs/.anycache_web")

In [115]:
# # #DEBUG clear
# import shutil
# shutil.rmtree(f_cache)


# shutil.rmtree(f_cache_web)

In [97]:
"""
HACK temporarily change sys.argv
"""
import sys
from typing import List


class Argv:
    def __init__(self, new_argv: List[str]):
        self.new_argv = new_argv
        self.original_argv = None

    def __enter__(self):
        self.original_argv = sys.argv[:]
        sys.argv[:] = self.new_argv
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        sys.argv[:] = self.original_argv

In [98]:
banned_titles = [
    "reddit - dive into anything",
    "just a moment...",
    "",
    "x.com",
    'x',
]
def is_banned_title(title):
    return title.lower().strip() in banned_titles

In [18]:
"""use lncrawl browser to get titles

also use a persistent header browser so we can manually solve cloudfare


note: if it fails to start browser, try restarting VSCODE
"""

# see https://github.com/dipu-bd/lightnovel-crawler/blob/master/lncrawl/core/app.py#L84
import logging

from lncrawl.core.browser import Browser
from lncrawl.core.exeptions import ScraperErrorGroup
from lncrawl.core.scraper import Scraper
from readability import Document


from lncrawl.core.browser import EC

# start a browser
with Argv([""]):
    browser = Browser(headless=False)
    browser._init_browser()
    browser._apply_cookies()

# manually pass cloudfare (YOU NEED TO CLICK!)
url = "https://www.fanfiction.net/s/10758358/1/What-You-Leave-Behind"
browser.visit(url)
browser.wait("body")
browser.wait(
    "#challenge-running",
    expected_conditon=EC.invisibility_of_element,
    timeout=30,
)
# reader = Document(browser.html)



In [116]:
from bs4 import BeautifulSoup
import time

# @anycache(f_cache_web)
def lncrawl_guess_novel_title(url: str) -> str:
    try:
        scraper = Scraper(url)
        response = scraper.get_response(url)
        reader = Document(response.text)
        title = reader.short_title()
        assert not is_banned_title(title), f"bad title `{title}` for url={url}"
    except (AssertionError, ) + ScraperErrorGroup as e:
        # if logger.isEnabledFor(logging.DEBUG):
        #     logger.exception("Failed to get response: %s", e)
        logger.debug(f"[lncrawl:scrape].failed url={url} with {e}, trying [lncrawl.browser]")
        browser.visit(url)
        browser.wait("body", timeout=40)        
        time.sleep(2) # wait for javascript to load
        reader = Document(browser.html)
        title = reader.short_title()
        assert not is_banned_title(title), f"bad title `{title}` for url={url}"
    return title


# # # Test
# urls = [
# 'https://x.com/AwfulFantasy/status/1866511632995962894',
# 'https://reddit.com/r/motheroflearning/comments/5v0zl0/links_to_discussion_threads',
# #     'https://parahumans.wordpress.com/',
# #     # 'https://www.wuxiaworld.com/novel/overgeared',
# #     'https://www.fanfiction.net/s/10758358/1/What-You-Leave-Behind',
# #     # 'https://www.royalroad.com/fiction/81002/the-years-of-apocalypse-a-time-loop-progression',
# ]

# # with Argv(['']):
# for url in urls:
#     r = lncrawl_guess_novel_title(url)
#     print(url)
#     print(r)

In [117]:
import cloudscraper
from bs4 import BeautifulSoup

session = cloudscraper.create_scraper()

@anycache(f_cache_web)
def cloudscrape_title(url):
    r = session.get(url, timeout=15, allow_redirects=True)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    title = soup.title.text.strip()
    assert not is_banned_title(title), f"bad title `{title}` for url={url}"
    return title

In [118]:
# dedup using title

from pathlib import Path

# TODO I should look at how fanficfare and lncrawler do this
# https://github.com/dipu-bd/lightnovel-crawler/blob/master/lncrawl/core/app.py#L84

@anycache(f_cache)
def get_title_md(url):
    t = titles_md.get(url, [])
    if t:
        s = pd.Series(t)
        title = s.value_counts().index[0]  # return most common title
        assert not is_banned_title(title), f"bad title `{title}` for url={url}"
        return title
    raise IndexError(f"Failed to get title for {url}")


def get_slugged_title(url2):
    url = url2

    if "reddit.com" in url:
        # if reddit thread get thead name e.g. 'https://www.reddit.com/r/Parahumans/comments/9oexpn/ward_spoilers_frustration_with_the_state_of_the/e7tprmt/?context=3'
        if "/comments/" in url:
            p = re.match(r".+reddit.com/r/([^/]+)/comments/([^/]+)/([^/]*)", url)
            if p:
                return 'thread ' + p.group(1).replace("_", " ") + " " + p.group(3).replace("_", " ")
        # 'https://reddit.com/r/motheroflearning/comments/5v0zl0/',
        if "/r/" in url:
            p = re.match(r".+reddit.com/r/([^/]+)/", url)
            if p:
                return 'r/' + p.group(1)


    # if that doesn't work, does the end of the url contain a slugged title?
    # e.g. https://www.fanfiction.net/s/5193644/harry-potter-and-the-methods-of-rationality
    slugged_title = url.split("/")[-1].replace("-", " ")
    if " " in slugged_title:
        title = slugged_title
        assert not is_banned_title(title), f"bad title `{title}` for url={url2}"
        return title
    raise ValueError(f"Failed to get title for {url2}")


# @anycache(f_cache)
def get_title(url):
    # first check if it's in the markdown cache
    try:
        return get_title_md(url)
    except Exception as e:
        logger.debug(f"[md] Failed to get title for {url} {e}, trying lncrawl")

    # try scraping from the web
    try:
        return lncrawl_guess_novel_title(url)
    except Exception as e:
        logger.debug(f"[lncrawl] Failed to get title for {url} {e}, trying cloudscrape")

    try:
        return cloudscrape_title(url)
    except Exception as e:
        logger.debug(f"[cloudscrape] Failed to get title for {url} {e}, trying slugged title")

    # fall back on slugged title?
    try:
        return get_slugged_title(url)
    except Exception as e:
        logger.debug(f"[slug] Failed to get title for {url} {e}")

    title = url

    assert not is_banned_title(title), f"bad title `{title}` for url={url}"

    return title


# url = df_links.index[0]

test_urls = [
    'https://x.com/AwfulFantasy/status/1866511632995962894',
    # 'https://reddit.com/r/motheroflearning/comments/5v0zl0/links_to_discussion_threads',
    # 'https://www.fimfiction.net/story/403715/zebric',
    # 'https://myanimelist.net/anime/18153/Kyoukai_no_',
    # 'https://yudkowsky.tumblr.com/',
    # "https://www.fanfiction.net/s/10758358/1/What-You-Leave-Behind",
    # "https://www.wuxiaworld.com/novel/overgeared",
    # "https://www.royalroad.com/fiction/81002/the-years-of-apocalypse-a-time-loop-progression",
]
for url in test_urls:
    r = get_title(url)
    print(f"{url} -> `{r}`")

2024-12-25 18:23:59.339 | DEBUG    | __main__:get_title:51 - [md] Failed to get title for https://x.com/AwfulFantasy/status/1866511632995962894 Failed to get title for https://x.com/AwfulFantasy/status/1866511632995962894, trying lncrawl
2024-12-25 18:23:59.812 | DEBUG    | __main__:lncrawl_guess_novel_title:15 - [lncrawl:scrape].failed url=https://x.com/AwfulFantasy/status/1866511632995962894 with bad title `x.com` for url=https://x.com/AwfulFantasy/status/1866511632995962894, trying [lncrawl.browser]


https://x.com/AwfulFantasy/status/1866511632995962894 -> `Awful Fantasy on X: "https://t.co/bXcCbRcmpI"`


In [119]:
browser.visit(url)
browser.wait("body", timeout=20)
reader = Document(browser.html)
title = reader.short_title()
title

'X'

In [120]:
# QC
urls = df_links3.reset_index().url.sample(20, random_state=136)
titles = urls.progress_map(get_title).values
pd.Series(urls.values, index=titles)

  0%|          | 0/20 [00:00<?, ?it/s]

2024-12-25 18:24:04.094 | DEBUG    | __main__:get_title:51 - [md] Failed to get title for https://ff.net Failed to get title for https://ff.net, trying lncrawl
2024-12-25 18:24:06.251 | DEBUG    | __main__:get_title:51 - [md] Failed to get title for https://www.smbc-comics.com/comic/2012-11-19 Failed to get title for https://www.smbc-comics.com/comic/2012-11-19, trying lncrawl
2024-12-25 18:24:07.184 | DEBUG    | __main__:get_title:51 - [md] Failed to get title for https://web.archive.org/web/20230609092523/https://old.reddit.com/r/apolloapp/comments/144f6xm/apollo_will_close_down_on_june_30th_reddits/ Failed to get title for https://web.archive.org/web/20230609092523/https://old.reddit.com/r/apolloapp/comments/144f6xm/apollo_will_close_down_on_june_30th_reddits/, trying lncrawl
2024-12-25 18:24:10.825 | DEBUG    | __main__:get_title:51 - [md] Failed to get title for https://blackadventurescomic.com/ Failed to get title for https://blackadventurescomic.com/, trying lncrawl
2024-12-25 1

恭喜，站点创建成功！                                                                                                                                                                                                                                                https://ff.net
Wanderlust                                                                                                                                                                                                                            http://royalroadl.com/fiction/3670
Saturday Morning Breakfast Cereal                                                                                                                                                                                           https://www.smbc-comics.com/comic/2012-11-19
📣 Apollo will close down on June 30th. Reddit’s recent decisions and actions have unfortunately made it impossible for Apollo to continue. Thank you so, so much for all the support over the years. ❤️ : apo

In [ ]:
# # HACK
# n = 10000 # DEV limit
# df_links3 = df_links3.iloc[:n].copy()

df_links3['title'] = df_links3.reset_index().url.progress_map(get_title).values

  0%|          | 0/10000 [00:00<?, ?it/s]

2024-12-25 18:24:19.354 | DEBUG    | __main__:get_title:51 - [md] Failed to get title for https://worththecandle.wikia.com/wiki/Worth_the_Candle_Wiki Failed to get title for https://worththecandle.wikia.com/wiki/Worth_the_Candle_Wiki, trying lncrawl
2024-12-25 18:24:20.026 | DEBUG    | __main__:get_title:51 - [md] Failed to get title for https://paperelemental.blogspot.com/2020/05/hexcoords.html Failed to get title for https://paperelemental.blogspot.com/2020/05/hexcoords.html, trying lncrawl
2024-12-25 18:24:22.689 | DEBUG    | __main__:get_title:51 - [md] Failed to get title for https://www.fictionpress.com/s/2961893/26/Mother-of-Learning Failed to get title for https://www.fictionpress.com/s/2961893/26/Mother-of-Learning, trying lncrawl
2024-12-25 18:24:23.154 | DEBUG    | __main__:lncrawl_guess_novel_title:15 - [lncrawl:scrape].failed url=https://www.fictionpress.com/s/2961893/26/Mother-of-Learning with 403 Client Error: Forbidden for url: https://www.fictionpress.com/s/2961893/26/

In [ ]:
# QC how many titles failed?
print(df_links3['title'].str.startswith('http').sum())
print(df_links3['title'].str.contains('just a moment').sum())
print(df_links3['title'].isna().sum())

In [ ]:
t = df_links3['title']
t[t.str.startswith('http')]

In [ ]:
# join by title
def join_uniq(x: list[str]):
    return "\n".join(set(x))


def chain_lists(x: list[list[str]]):
    return [item for sublist in x for item in sublist]


df3 = (
    df_links3.reset_index().groupby("title")
    .agg(
        {
            "score": "sum",
            "n_links": "sum",
            'n_comments': 'sum',
            "comments": chain_lists,
            "thread_urls": chain_lists,
            'first_link_utc': 'min',
            'last_link_utc': 'max',
            'url': list,
        }
    )
    .sort_values("score", ascending=False)
)
df3.head(33)

In [ ]:
print(f"{len(df_links3)} -> {len(df3)} after title dedup")

### Export to html

In [57]:
def format_flair(author_flair_text):
    if author_flair_text:
        return f" <em>{author_flair_text}</em>"
    return ""

import markdown
def commentmd2html(x: dict) -> str:
    body = markdown.markdown(x['body'])
    ts = pd.to_datetime(x['created_utc'], unit='s').strftime('%Y-%m-%d')
    flair = format_flair(x['author_flair_text'])
    url = prefix + x['permalink']
    s = f"""<h3><a href="{url}">{x.get('author', 'anon')} [{x['score']:+}] {flair} <sup>{ts}</sup></a></h3>
{body}
"""
    # print(s)
    return s

def collapsibe(title, body):
    return f"""<details><summary>{title}</summary>
{body}
</details>
"""

prefix = "https://reddit.com"
def c2md(x):
    return collapsibe(x['id'], commentmd2html(x))

# # QC test
# x = df3.iloc[0].comments[0]
# from IPython.display import display, HTML
# display(HTML(c2md(x)))

In [ ]:
# First transform the fields for display
d = df3.reset_index().sort_values("score", ascending=False)


def url2a(url):
    text = url
    if "reddit.com/r/rational" in url:
        text = url.split("/")[-2]
        # text = url.replace('https://reddit.com/r/rational/comments/', '')

    return f'<a href="{url}">{text}</a>'


def urls2a(urls, sep="<br>"):
    if isinstance(urls, str):
        urls = urls.split("\n")

    urls = list(set(urls))

    return sep.join(url2a(u) for u in urls)


d["url"] = d["url"].apply(lambda x: collapsibe("fiction_urls", urls2a(x)))
d["score"] = d["score"].round(2)

prefix = 'https://reddit.com/r/rational/comments/'
d["comment_urls"] = d['comments'].apply(lambda x: collapsibe("comment_links", urls2a([prefix+y['permalink'] for y in x], sep=" ")))
d["thread_urls"] = d['thread_urls'].apply(lambda x: collapsibe("threads", urls2a(x, sep=" ")))
d['comments'] = d['comments'].progress_apply(lambda x: collapsibe("comments", "<br>".join([c2md(c) for c in x])))


In [ ]:
# put into the html template
import jinja2

environment = jinja2.Environment()
template = open("../index.jinja2.html").read()
template = environment.from_string(template)


data = d.to_json(orient="values")

hidden = ["comments", "comment_urls"]
columns = [
    {
        "title": c,
        "visible": c not in hidden,
        "searchable": c not in hidden,
    }
    for c in d.columns
]
columns = json.dumps(columns)


html = template.render(
    data=data,
    columns=columns,
)
html_out = Path("../outputs/index.html").resolve()
open(html_out, "w").write(html)
columns

In [ ]:
from IPython.display import HTML, display

htmla = f'<a href="{html_out}">View the page {html_out}</a>'
display(HTML(htmla))

In [61]:
# df3.to_html('../outputs/links3.html')

## Extra get a llm summary of each link [WIP]

Grab all md's that mention a story, ask claude to summarize

We could also get total karma per mention

In [62]:
import dotenv

dotenv.load_dotenv()
from openai import OpenAI

client = OpenAI()

import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")
cost = 0.150 / 1e6

In [63]:
md_posts = []
fs = sorted(Path("../data/cache2").glob("*.md"))
for f in fs:
    s = f.open().read()
    md_posts.append(s)


def get_post_context(urls: List[str], char_budget=100000):
    # TODO maybe I should just get comment with links, and children?


    # FIXME take in multiple urls and titles
    # FIXME given a token or char budget
    assert len(urls) > 0
    # from comments
    # return df3.loc[title].comments

    # or I could just all markdowns with

    # TODO use langchain chunking?

    matches = []
    for ii, post in enumerate(md_posts):
        for url in urls:
            if url in post:
                matches.append(post)
                break

    budget_pp = char_budget / len(matches)
    s = ""
    for i in range(len(matches)):
        post = matches[i]

        for url in urls:
            if url in post:
                ind = post.index(url)

        i0 = int(max(0, ind - budget_pp // 4))
        i1 = int(min(len(post), ind + budget_pp // 4 * 3))
        post_chunk = post[i0:i1]
        if i0 > 0:
            post_chunk = "..." + post_chunk
        if i1 < len(post):
            post_chunk = post_chunk + "..."

        s += f"\n\n----- Thread {ii} -----\n\n" + post_chunk
    return s


# url = df3.url[0].split('\n')
# print(url)
# c = get_context(url, 400000)
# print(c[:1000])

In [64]:
from typing import List, Optional

from pydantic import BaseModel, Field


class FictionInfo(BaseModel):
    title: str
    description: str = Field(description="Brief, very concise, description of the work")
    tags: List[str] = Field(
        description="""Long list of descriptors: format (web serial, fanfic, lightnovel, short, complete, comic), genre (scifi, fantasy)
        Key elements (rational, timeloop, litrpg, progression, cultivation, isekai)
        Content notes (grimdark, romance, harem, queer, funny, NSFW)
        """
    )

    # status: Optional[str] = Field(description="complete/ongoing/hiatus/abandoned")
    # type: str = Field(description='e.g. fanfiction, original, comic, etc.')

    reviews_quotes: List[str] = Field(
        description="Directly and fully quote exerpts from each users' comments about the fiction"
    )
    reviews_summary: Optional[str] = Field(
        description="Structured summary of reviews including 1) what aspects users comment on, 2) why users recommend it 3) disrecommend it, 4) how many users like vs dislike it, etc."
    )

    quality: float = Field(
        # ge=0.0, le=10.0,
        description="Overall user sentiment out of 10"
    )
    rationality: Optional[float] = Field(
        # ge=0.0, le=10.0,
        description="Systematic worldbuilding, character competence, logical consistency. Where HPMOR is a 10 and Worm is a 5."
    )
    rating_writing: Optional[float]
    rating_plot: Optional[float]
    rating_character: Optional[float]
    rating_worldbuilding: Optional[float]


f_cache = Path("../outputs/.anycache3")


@anycache(f_cache)
def get_llm_summary(name: str, context: str):
    chat_completion = client.beta.chat.completions.parse(
        messages=[
            {
                "role": "system",
                "content": "You are Gwern Branwern, an internet librarian who specializes in rational fiction. You are summarising community reccomendations into a structured form.",
            },
            {
                "role": "user",
                "content": f"""For u/gwern please summarize structured information about {name}. Quote users in full and attribute the username if known.

### Context:

{context}""",
            },
        ],
        model="gpt-4o-mini",
        response_format=FictionInfo,
    )

    return chat_completion.choices[0].message.parsed.__dict__

In [ ]:
from openai.lib._pydantic import to_strict_json_schema

to_strict_json_schema(FictionInfo)

In [66]:
from IPython.display import display

In [ ]:
llm_info = []


l = min(1000, len(df3))
for i in tqdm(range(1, l)):
    title = df3.index[i]
    urls = df3.url[i]

    context = get_post_context(urls, char_budget=50000)
    tokens = len(enc.encode(context))
    print(
        f"Input Tokens: {tokens}. Input Cost: {cost * tokens:.4f} USD, for url {urls}"
    )

    llm_data = get_llm_summary(urls, context)

    llm_data["title2"] = title
    llm_data["url"] = urls

    # print(f"Content: {context}")
    # print(url, d)
    # display(llm_data)

    llm_info.append(llm_data)

    # 1/0

In [ ]:
urls

In [ ]:
df_llm = pd.DataFrame(llm_info)
df_llm
# also join with df4